In [3]:
import pandas as pd

def process_and_fix_data(input_file, output_file):
    try:
        # 1. Đọc file CSV
        df = pd.read_csv(input_file)
        print("Đang xử lý file...")

        # 2. Hàm xử lý logic: Khôi phục Title gốc và giữ Comment đã clean
        def fix_input_text(row):
            # Lấy title gốc (chưa bị ẩn danh <person>)
            raw_title = str(row['raw_title']) if pd.notna(row['raw_title']) else ""
            # Xóa xuống dòng dư thừa trong title
            raw_title = raw_title.replace('\n', ' ').strip()
            
            # Lấy input_text hiện tại
            current_input = str(row['input_text']) if pd.notna(row['input_text']) else ""
            
            # Tách phần comment ra (dựa vào dấu phân cách </s>)
            parts = current_input.split(' </s> ', 1)
            
            comment_part = ""
            if len(parts) > 1:
                comment_part = parts[1] # Lấy phần sau dấu </s> (là comment đã clean teencode)
            else:
                # Trường hợp không có dấu ngăn cách, giữ nguyên input cũ làm comment
                comment_part = current_input
            
            # Ưu tiên dùng Title gốc (raw_title) nếu có
            if raw_title:
                title_part = raw_title
            elif len(parts) > 1:
                title_part = parts[0] # Fallback về title cũ nếu không có raw
            else:
                title_part = ""
                
            # Ghép lại: Title Gốc + </s> + Comment đã clean
            if title_part and comment_part:
                return f"{title_part} </s> {comment_part}"
            elif title_part:
                return title_part
            elif comment_part:
                return comment_part
            else:
                return ""

        # Áp dụng logic vào cột mới
        df['input_text'] = df.apply(fix_input_text, axis=1)

        # 3. Sắp xếp lại cột: Đưa 'label' về ngay sau 'input_text' cho dễ gán nhãn
        cols = list(df.columns)
        if 'label' in cols and 'input_text' in cols:
            cols.remove('label')
            input_idx = cols.index('input_text')
            cols.insert(input_idx + 1, 'label')
            df = df[cols]

        # 4. Lưu file mới
        df.to_csv(output_file, index=False, encoding='utf-8-sig')
        print(f"✅ Đã xong! File mới được lưu tại: {output_file}")
        
    except Exception as e:
        print(f"❌ Có lỗi xảy ra: {e}")

# Chạy hàm
if __name__ == "__main__":
    process_and_fix_data('IAA_set_500_samples.csv', 'IAA_set_500_samples_fixed_title.csv')

Đang xử lý file...
✅ Đã xong! File mới được lưu tại: IAA_set_500_samples_fixed_title.csv


In [1]:
import pandas as pd
import json
import re

def map_labels_from_json_to_csv(json_path, csv_path, output_path):
    # 1. Load dữ liệu JSON
    with open(json_path, 'r', encoding='utf-8') as f:
        json_data = json.load(f)

    # 2. Tạo dictionary mapping: {nội dung câu -> nhãn số}
    # Sử dụng dict để tra cứu cực nhanh
    label_map = {}
    for item in json_data:
        text = item['data'].get('comment', '').strip()
        label = None
        
        # Trích xuất nhãn từ phần annotations
        if item.get('annotations') and len(item['annotations']) > 0:
            results = item['annotations'][0].get('result', [])
            for res in results:
                if res.get('from_name') == 'label':
                    choices = res.get('value', {}).get('choices', [])
                    if choices:
                        # Lấy số từ nhãn (ví dụ: "2 - Thù ghét" -> 2)
                        match = re.search(r'\d', str(choices[0]))
                        if match:
                            label = int(match.group())
                            break
        
        if label is not None:
            label_map[text] = label

    # 3. Load file CSV
    df = pd.read_csv(csv_path)

    # 4. Hàm logic để gán nhãn
    def get_new_label(row):
        # Ưu tiên khớp bằng cleaned_comment (do Label Studio thường lưu text đã clean)
        cleaned = str(row['cleaned_comment']).strip()
        if cleaned in label_map:
            return label_map[cleaned]
        
        # Thử khớp bằng raw_comment nếu cleaned không thấy
        raw = str(row['raw_comment']).strip()
        if raw in label_map:
            return label_map[raw]
        
        # "Cái nào không có khỏi gán" -> Giữ nguyên giá trị cũ trong CSV
        return row['label']

    # Thực hiện cập nhật cột label
    df['label'] = df.apply(get_new_label, axis=1)

    # 5. Lưu kết quả
    df.to_csv(output_path, index=False, encoding='utf-8-sig')
    
    assigned_count = df['label'].notna().sum()
    print(f"--- Hoàn tất ---")
    print(f"Đã cập nhật nhãn thành công. Hiện có {assigned_count} dòng có nhãn.")
    print(f"File đã lưu tại: {output_path}")

# Chạy script với file của bạn
map_labels_from_json_to_csv(
    'final_thien_gold_sample.json', 
    'labeling_task_Thien.csv', 
    'labeling_task_Thien_assigned.csv'
)

--- Hoàn tất ---
Đã cập nhật nhãn thành công. Hiện có 1842 dòng có nhãn.
File đã lưu tại: labeling_task_Thien_assigned.csv


In [1]:
import pandas as pd
import json
import numpy as np

# 1. Đọc file CSV
df = pd.read_csv('IAA_set_500_samples.csv')

# 2. Hàm tạo cấu trúc JSON cho Label Studio
def create_label_studio_task(row):
    # Dữ liệu hiển thị (Input Data)
    task = {
        "data": {
            "raw_title": row['raw_title'],
            "raw_comment": row['raw_comment'],
            "source_platform": row['source_platform'],
            "likes": int(row['likes']),
            "replies_count": int(row['replies_count'])
        }
    }
    
    # Xử lý nhãn cũ (Pre-annotation)
    # Chỉ tạo annotation nếu dòng đó đã có label (không bị NaN)
    if pd.notna(row['label']):
        # Chuyển 1.0 -> "1", 0.0 -> "0"
        label_val = str(int(row['label']))
        
        # Cấu trúc Annotation của Label Studio
        annotation = {
            "result": [
                {
                    "from_name": "label",   # Tên biến Choices trong XML
                    "to_name": "comment",   # Tên biến Text cần gán trong XML
                    "type": "choices",
                    "value": {
                        "choices": [label_val] # Giá trị nhãn được chọn
                    }
                }
            ]
        }
        task["annotations"] = [annotation]
        
    return task

# 3. Áp dụng và xuất file
json_output = df.apply(create_label_studio_task, axis=1).tolist()

output_filename = 'thanhthien_import_with_labels.json'
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(json_output, f, ensure_ascii=False, indent=2)

print(f"✅ Đã tạo xong file: {output_filename}")
print(f"Tổng số mẫu: {len(json_output)}")

✅ Đã tạo xong file: thanhthien_import_with_labels.json
Tổng số mẫu: 500


In [5]:
import pandas as pd
import json
import numpy as np

# 1. Đọc file Excel
df = pd.read_excel('IAA_set_500_samples.xlsx')

print(f"📊 Tổng số dòng: {len(df)}")
print(f"📋 Các cột trong file: {list(df.columns)}")

# 2. Xác định các cột label (tất cả cột có tên chứa 'label')
label_columns = [col for col in df.columns if 'label' in col.lower()]
print(f"\n🏷️ Phát hiện {len(label_columns)} cột label: {label_columns}")

# 3. Hàm tạo cấu trúc JSON cho Label Studio
def create_label_studio_task(row):
    # === PHẦN 1: DATA (hiển thị) ===
    task = {
        "data": {
            "raw_title": str(row['raw_title']) if pd.notna(row['raw_title']) else "",
            "raw_comment": str(row['raw_comment']) if pd.notna(row['raw_comment']) else "",
            "source_platform": str(row['source_platform']) if pd.notna(row['source_platform']) else "",
            "likes": int(row['likes']) if pd.notna(row['likes']) else 0,
            "replies_count": int(row['replies_count']) if pd.notna(row['replies_count']) else 0
        }
    }
    
    # === PHẦN 2: PREDICTIONS (các label đã gán) ===
    # Dùng predictions thay vì annotations để tránh lỗi completed_by
    predictions = []
    
    for label_col in label_columns:
        if pd.notna(row[label_col]):
            try:
                # Chuyển đổi label value
                label_val = str(int(float(row[label_col])))
                
                # Tạo 1 prediction cho mỗi cột label
                prediction = {
                    "model_version": label_col,  # Dùng tên cột làm model version
                    "result": [
                        {
                            "from_name": "label",
                            "to_name": "comment",
                            "type": "choices",
                            "value": {
                                "choices": [label_val]
                            }
                        }
                    ]
                }
                predictions.append(prediction)
            except (ValueError, TypeError):
                print(f"⚠️ Không thể chuyển đổi giá trị tại cột '{label_col}', dòng {row.name}")
    
    # Chỉ thêm predictions nếu có ít nhất 1 label
    if predictions:
        task["predictions"] = predictions
    
    # === PHẦN 3: META (thông tin bổ sung - không bắt buộc) ===
    # Lưu tất cả các cột khác để tham khảo
    meta_data = {}
    for col in df.columns:
        if col not in ['raw_title', 'raw_comment', 'source_platform', 'likes', 'replies_count'] and col not in label_columns:
            if pd.notna(row[col]):
                meta_data[col] = str(row[col])
    
    if meta_data:
        task["data"]["meta"] = meta_data
    
    return task

# 4. Áp dụng và xuất file
try:
    json_output = df.apply(create_label_studio_task, axis=1).tolist()
    
    # === XUẤT FILE JSON ===
    output_filename = 'labelstudio_import_with_all_labels.json'
    with open(output_filename, 'w', encoding='utf-8') as f:
        json.dump(json_output, f, ensure_ascii=False, indent=2)
    
    # === THỐNG KÊ ===
    print(f"\n✅ Đã tạo xong file: {output_filename}")
    print(f"📦 Tổng số task: {len(json_output)}")
    
    # Đếm số predictions
    total_predictions = sum(len(task.get("predictions", [])) for task in json_output)
    with_labels = sum(1 for task in json_output if "predictions" in task)
    
    print(f"🏷️ Số task có pre-annotation: {with_labels}")
    print(f"🏷️ Tổng số predictions: {total_predictions}")
    print(f"📝 Số task chưa có label: {len(json_output) - with_labels}")
    
    # Hiển thị mẫu
    print(f"\n🔍 Mẫu task đầu tiên:")
    print(json.dumps(json_output[0], indent=2, ensure_ascii=False))
    
    # Thống kê theo annotator
    print(f"\n📊 Thống kê theo annotator:")
    for label_col in label_columns:
        count = df[label_col].notna().sum()
        print(f"  - {label_col}: {count} labels")
    
except Exception as e:
    print(f"❌ Lỗi: {str(e)}")
    import traceback
    traceback.print_exc()

📊 Tổng số dòng: 500
📋 Các cột trong file: ['id', 'input_text', 'label', 'raw_comment', 'raw_title', 'cleaned_comment', 'cleaned_title', 'source_platform', 'source_url', 'post_id', 'video_id', 'timestamp', 'username', 'likes', 'replies_count', 'char_length', 'word_count', 'has_emoji', 'topic', 'note', 'labeler', 'confidence', 'is_hate_keyword', 'sampling_strategy']

🏷️ Phát hiện 2 cột label: ['label', 'labeler']

✅ Đã tạo xong file: labelstudio_import_with_all_labels.json
📦 Tổng số task: 500
🏷️ Số task có pre-annotation: 291
🏷️ Tổng số predictions: 291
📝 Số task chưa có label: 209

🔍 Mẫu task đầu tiên:
{
  "data": {
    "raw_title": "Vụ xe Phân khối lớn đâm vào đôi bạn trẻ trên phố Trịnh Văn Bô: Video hành trình ghi lại cảnh chiếc BMW phóng vơi tốc độ kinh hoàng‼️\nNghe tiếng người bạn trai ôm bạn gái bị đứt lìa chân khóc thảm...",
    "raw_comment": "Nguyễn Văn Có thề chứ dcm tao cực ghéc mấy thể loại chạy pkl thể hiện như này dcm bảo sao",
    "source_platform": "Facebook",
    "likes

In [7]:
import json

# Đọc file bạn đã upload
input_file = 'labelstudio_import_with_all_labels.json'
output_file = 'labelstudio_ready_to_import.json'

try:
    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    valid_count = 0
    fixed_data = []

    for task in data:
        # Lấy thông tin data gốc
        new_task = {"data": task["data"]}
        
        # Tìm nhãn (ưu tiên lấy trong predictions nếu chưa có annotations)
        source_result = []
        
        # Check Predictions
        if task.get("predictions"):
            source_result = task["predictions"][0]["result"]
        # Check Annotations (nếu có)
        elif task.get("annotations"):
            source_result = task["annotations"][0]["result"]
            
        # Nếu tìm thấy kết quả gán nhãn, tạo Annotation chuẩn
        if source_result:
            # Đảm bảo value là list string (Vd: ["1"] thay vì [1])
            for item in source_result:
                if "value" in item and "choices" in item["value"]:
                    choices = item["value"]["choices"]
                    item["value"]["choices"] = [str(c) for c in choices]

            new_task["annotations"] = [{
                "result": source_result,
                "was_cancelled": False,
                "ground_truth": True  # Đánh dấu là nhãn chuẩn
            }]
            valid_count += 1
        
        fixed_data.append(new_task)

    # Lưu file mới
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(fixed_data, f, ensure_ascii=False, indent=2)

    print(f"✅ Đã xử lý xong! File mới: {output_file}")
    print(f"📊 Tổng số mẫu có nhãn: {valid_count} / {len(data)}")

except Exception as e:
    print(f"❌ Lỗi: {e}")

✅ Đã xử lý xong! File mới: labelstudio_ready_to_import.json
📊 Tổng số mẫu có nhãn: 291 / 500


# Check kappa 2 người

In [9]:
import pandas as pd

# 1. Khai báo tên file với đúng định dạng
file_huy = "Huy_500.csv"                # File CSV
file_ket_qua = "ket_qua_nop_bai.xlsx"   # File Excel
file_kiet = "kiet_final.xlsx"           # File Excel

# 2. Đọc dữ liệu (Sử dụng hàm tương ứng với định dạng)
# Lưu ý: Cần cài đặt thư viện openpyxl để đọc file Excel: pip install openpyxl
df_huy = pd.read_csv(file_huy)
df_ket_qua = pd.read_excel(file_ket_qua)
df_kiet = pd.read_excel(file_kiet)

# 3. Chuẩn hóa và lọc các cột cần thiết
# File Huy: label
df_huy_sub = df_huy[['id', 'raw_comment', 'label']].rename(columns={'label': 'label_Huy'})

# File của bạn (ket_qua_nop_bai): label_final
df_ket_qua_sub = df_ket_qua[['id', 'label_final']].rename(columns={'label_final': 'label_Ban'})

# File Kiệt: label
df_kiet_sub = df_kiet[['id', 'label']].rename(columns={'label': 'label_Kiet'})

# 4. Gộp (Merge) 3 bảng dựa trên ID
merged_df = pd.merge(df_huy_sub, df_ket_qua_sub, on='id', how='inner')
merged_df = pd.merge(merged_df, df_kiet_sub, on='id', how='inner')

# 5. Làm sạch dữ liệu nhãn (Bỏ dòng trống và ép kiểu về số nguyên)
merged_df = merged_df.dropna(subset=['label_Huy', 'label_Ban', 'label_Kiet'])
merged_df['label_Huy'] = merged_df['label_Huy'].astype(int)
merged_df['label_Ban'] = merged_df['label_Ban'].astype(int)
merged_df['label_Kiet'] = merged_df['label_Kiet'].astype(int)

# 6. Lọc các dòng có sự khác biệt (Bất đồng giữa ít nhất 1 người)
disagreements = merged_df[
    (merged_df['label_Huy'] != merged_df['label_Ban']) |
    (merged_df['label_Huy'] != merged_df['label_Kiet']) |
    (merged_df['label_Ban'] != merged_df['label_Kiet'])
]

# 7. Xuất kết quả ra file Excel để dễ thảo luận nhóm
disagreements.to_excel('cac_comment_can_thong_nhat.xlsx', index=False)

print(f"Tổng số mẫu chung: {len(merged_df)}")
print(f"Số mẫu cần thảo luận lại: {len(disagreements)}")

Tổng số mẫu chung: 499
Số mẫu cần thảo luận lại: 190


In [1]:
import pandas as pd
import json

# 1. Đọc file
df = pd.read_csv('labeling_task_Thien.csv')

tasks = []
for index, row in df.iterrows():
    if pd.notna(row['input_text']):
        text_content = str(row['input_text'])
        
        # Tách Context và Comment dựa trên '</s>'
        if '</s>' in text_content:
            parts = text_content.split('</s>')
            context_val = parts[0].strip()
            comment_val = " ".join(parts[1:]).strip() # Gộp lại nếu có nhiều hơn 1 dấu </s>
        else:
            # Trường hợp không có dấu phân cách
            context_val = "Không có ngữ cảnh"
            comment_val = text_content

        # Tạo task data
        task = {
            "data": {
                "context": context_val,
                "comment": comment_val
            }
        }

        # Xử lý Pre-label (Nhãn có sẵn)
        if pd.notna(row['label']):
            label_val = str(int(row['label']))
            task["predictions"] = [{
                "model_version": "v1_pre_labeled",
                "result": [{
                    "from_name": "label",
                    "to_name": "comment_text", # Link tới phần comment text
                    "type": "choices",
                    "value": { "choices": [label_val] }
                }]
            }]
        
        # Xử lý Note (nếu có)
        if 'note' in row and pd.notna(row['note']):
             task["predictions"][0]["result"].append({
                "from_name": "note",
                "to_name": "comment_text",
                "type": "textarea",
                "value": { "text": [str(row['note'])] }
            })

        tasks.append(task)

# 2. Xuất file JSON
with open('tasks_split_context.json', 'w', encoding='utf-8') as f:
    json.dump(tasks, f, ensure_ascii=False, indent=2)

print("Đã tạo file 'tasks_split_context.json'.")

Đã tạo file 'tasks_split_context.json'.


# Test protonx

In [7]:
import pandas as pd
import re
from underthesea import text_normalize
from tqdm import tqdm

# =====================================================
# 1. CẤU HÌNH TỪ ĐIỂN (HIỆU CHỈNH THEO CHIẾN THUẬT 6.2)
# =====================================================

# Từ điển Teencode thông dụng (Lớp phủ bề mặt)
TEENCODE_DICT = {
    "k": "không", "ko": "không", "bik": "biết", "ns": "nói",
    "j": "gì", "v": "vậy", "z": "vậy", "đc": "được", "hnay": "hôm nay",
    "m": "mày", "t": "tao", "vcl": "vãi cả lồn", "vkl": "vãi cả lồn"
}

# Từ điển "Độc hại ẩn dụ" (Lớp giải mã chuyên sâu)
EXPERT_TOXIC_DICT = {
    "hit như đạt": "hát như địt",
    "nốn lừng": "nứng lồn",
    "paki": "bắc kỳ",
    "namky": "nam kỳ",
    "lẹo cái": "lgbt",
    "ăn mặn đái khai": "mất dạy"
}

# Danh sách bảo vệ (Safety Gate) - Tuyệt đối không để Regex Person đụng vào
PROTECTED_KEYWORDS = ["CHÓ", "LỢN", "NGU", "ĐIÊN", "BẮC KỲ", "NAM KỲ", "LGBT"]

# =====================================================
# 2. HÀM XỬ LÝ CHÍNH
# =====================================================

def hybrid_clean_text(text):
    if not isinstance(text, str) or not text:
        return ""

    # BƯỚC 1: Chuẩn hóa Unicode bằng Underthesea (ProtonX's choice)
    text = text_normalize(text)

    # BƯỚC 2: Bảo vệ "Vạch kẻ đỏ" (Vụ CON CHÓ)
    # Tạm ẩn các từ nhạy cảm để không bị các hàm xử lý sau làm biến dạng
    for word in PROTECTED_KEYWORDS:
        text = re.sub(rf'\b{word}\b', f"__SAFE_{word}__", text, flags=re.IGNORECASE)

    # BƯỚC 3: Giải mã Coded Toxicity (Nói lái như "hit như đạt")
    # Phải làm bước này trước khi split từ đơn
    for coded, original in EXPERT_TOXIC_DICT.items():
        text = text.lower().replace(coded, original)

    # BƯỚC 4: Dịch Teencode từ đơn
    words = text.split()
    cleaned_words = [TEENCODE_DICT.get(w.lower(), w) for w in words]
    text = " ".join(cleaned_words)

    # BƯỚC 5: Xử lý lặp ký tự (nguuuu -> ngu)
    text = re.sub(r'([a-z])\1{2,}', r'\1', text)

    # BƯỚC 6: Giải phóng từ bảo vệ
    for word in PROTECTED_KEYWORDS:
        text = text.replace(f"__SAFE_{word}__", word)

    return text.strip()

# =====================================================
# 3. CHẠY TRÊN FILE XLSX
# =====================================================

def process_xlsx(input_file, output_file, column_name):
    print(f"🚀 Đang đọc file: {input_file}")
    df = pd.read_excel(input_file)

    if column_name not in df.columns:
        print(f"❌ Lỗi: Không tìm thấy cột '{column_name}' trong file!")
        return

    print("🛠️ Đang thực hiện Hybrid Cleaning...")
    # Dùng tqdm để theo dõi tiến độ (quan trọng khi làm 3.000 dòng)
    tqdm.pandas()
    df['cleaned_content'] = df[column_name].progress_apply(hybrid_clean_text)

    print(f"💾 Đang lưu kết quả tại: {output_file}")
    df.to_excel(output_file, index=False)
    print("✅ Hoàn thành!")

# --- CẤU HÌNH ĐƯỜNG DẪN ---
INPUT_PATH = "youtube_comment_craw.xlsx"  # Tên file gốc của bạn
OUTPUT_PATH = "data_cleaned_raw_youtube.xlsx" # Tên file sau khi clean
TARGET_COL = "comment"         # Tên cột chứa bình luận

if __name__ == "__main__":
    process_xlsx(INPUT_PATH, OUTPUT_PATH, TARGET_COL)

🚀 Đang đọc file: youtube_comment_craw.xlsx
🛠️ Đang thực hiện Hybrid Cleaning...


100%|██████████| 18476/18476 [00:05<00:00, 3494.67it/s]


💾 Đang lưu kết quả tại: data_cleaned_raw_youtube.xlsx
✅ Hoàn thành!


In [ ]:
import pandas as pd

# Load the three files
df_huy = pd.read_csv('Huy_500.csv')
df_kiet = pd.read_csv('kiet_final.xlsx - Worksheet.csv')
df_final = pd.read_csv('ket_qua_nop_bai.xlsx - Sheet1.csv')

# Inspect df_kiet
print("File: kiet_final.xlsx - Worksheet.csv Info:")
df_kiet.info()
print(df_kiet[['id', 'label']].head())

# Alignment and Kappa calculation
# Merge all three on 'id'
merged = pd.merge(df_huy[['id', 'label']], df_kiet[['id', 'label']], on='id', suffixes=('_huy', '_kiet'))
merged = pd.merge(merged, df_final[['id', 'label_final']], on='id')

# Rename for clarity
merged.columns = ['id', 'label_huy', 'label_kiet', 'label_final']

# Drop NaNs
merged = merged.dropna()

# Convert to int
merged['label_huy'] = merged['label_huy'].astype(int)
merged['label_kiet'] = merged['label_kiet'].astype(int)
merged['label_final'] = merged['label_final'].astype(int)

from sklearn.metrics import cohen_kappa_score

kappa_huy_kiet = cohen_kappa_score(merged['label_huy'], merged['label_kiet'])
kappa_huy_final = cohen_kappa_score(merged['label_huy'], merged['label_final'])
kappa_kiet_final = cohen_kappa_score(merged['label_kiet'], merged['label_final'])

print(f"\nMerged Samples: {len(merged)}")
print(f"Kappa (Huy vs Kiet): {kappa_huy_kiet}")
print(f"Kappa (Huy vs Final): {kappa_huy_final}")
print(f"Kappa (Kiet vs Final): {kappa_kiet_final}")

# Calculate Fleiss' Kappa
import numpy as np

def fleiss_kappa(M):
    """Computes Fleiss' kappa for group of annotators.
    M: (N x k) matrix of counts of assignments to each of k categories by N raters.
    But here we have N samples and 3 raters.
    We need to transform our data into a matrix where rows are samples and columns are categories.
    """
    N, n_raters = M.shape
    n_categories = int(np.max(M)) + 1
    
    counts = np.zeros((N, n_categories))
    for i in range(N):
        for j in range(n_raters):
            counts[i, int(M[i, j])] += 1
            
    # Matrix of counts per category per subject
    M = counts
    N, k = M.shape
    n = np.sum(M[0, :])
    
    p = np.sum(M, axis=0) / (N * n)
    P_e = np.sum(p**2)
    
    P = (np.sum(M**2, axis=1) - n) / (n * (n - 1))
    P_bar = np.sum(P) / N
    
    kappa = (P_bar - P_e) / (1 - P_e)
    return kappa

ratings = merged[['label_huy', 'label_kiet', 'label_final']].values
f_kappa = fleiss_kappa(ratings)
print(f"Fleiss' Kappa (All 3): {f_kappa}")import pandas as pd

# Load the three files
df_huy = pd.read_csv('Huy_500.csv')
df_kiet = pd.read_csv('kiet_final.xlsx - Worksheet.csv')
df_final = pd.read_csv('ket_qua_nop_bai.xlsx - Sheet1.csv')

# Inspect df_kiet
print("File: kiet_final.xlsx - Worksheet.csv Info:")
df_kiet.info()
print(df_kiet[['id', 'label']].head())

# Alignment and Kappa calculation
# Merge all three on 'id'
merged = pd.merge(df_huy[['id', 'label']], df_kiet[['id', 'label']], on='id', suffixes=('_huy', '_kiet'))
merged = pd.merge(merged, df_final[['id', 'label_final']], on='id')

# Rename for clarity
merged.columns = ['id', 'label_huy', 'label_kiet', 'label_final']

# Drop NaNs
merged = merged.dropna()

# Convert to int
merged['label_huy'] = merged['label_huy'].astype(int)
merged['label_kiet'] = merged['label_kiet'].astype(int)
merged['label_final'] = merged['label_final'].astype(int)

from sklearn.metrics import cohen_kappa_score

kappa_huy_kiet = cohen_kappa_score(merged['label_huy'], merged['label_kiet'])
kappa_huy_final = cohen_kappa_score(merged['label_huy'], merged['label_final'])
kappa_kiet_final = cohen_kappa_score(merged['label_kiet'], merged['label_final'])

print(f"\nMerged Samples: {len(merged)}")
print(f"Kappa (Huy vs Kiet): {kappa_huy_kiet}")
print(f"Kappa (Huy vs Final): {kappa_huy_final}")
print(f"Kappa (Kiet vs Final): {kappa_kiet_final}")

# Calculate Fleiss' Kappa
import numpy as np

def fleiss_kappa(M):
    """Computes Fleiss' kappa for group of annotators.
    M: (N x k) matrix of counts of assignments to each of k categories by N raters.
    But here we have N samples and 3 raters.
    We need to transform our data into a matrix where rows are samples and columns are categories.
    """
    N, n_raters = M.shape
    n_categories = int(np.max(M)) + 1
    
    counts = np.zeros((N, n_categories))
    for i in range(N):
        for j in range(n_raters):
            counts[i, int(M[i, j])] += 1
            
    # Matrix of counts per category per subject
    M = counts
    N, k = M.shape
    n = np.sum(M[0, :])
    
    p = np.sum(M, axis=0) / (N * n)
    P_e = np.sum(p**2)
    
    P = (np.sum(M**2, axis=1) - n) / (n * (n - 1))
    P_bar = np.sum(P) / N
    
    kappa = (P_bar - P_e) / (1 - P_e)
    return kappa

ratings = merged[['label_huy', 'label_kiet', 'label_final']].values
f_kappa = fleiss_kappa(ratings)
print(f"Fleiss' Kappa (All 3): {f_kappa}")

# Teencode
"https://huggingface.co/protonx-models/distilled-protonx-legal-tc" bạn có chạy đúng qua model này không

In [ ]:
import pandas as pd
from transformers import pipeline
import torch
from tqdm import tqdm

print("⏳ Đang tải Model...")
# Dùng model phù hợp hơn cho tiếng Việt
corrector = pipeline(
    "text2text-generation",
    model="VietAI/vit5-base-vietnews-summarization",  # Hoặc thử vinai/bartpho-syllable
    device=0 if torch.cuda.is_available() else -1
)

# Từ điển teencode → chuẩn
TEENCODE_MAPPING = {
    "nh th": "nhiều thằng",
    "mk": "mình",
    "tbm": "trong khi đó",
    "hit như đạt": "hát như địt",
    "vs": "với",
    "dc": "được",
    "ko": "không",
    "k": "không",
    "j": "gì",
    "cx": "cũng",
    "trc": "trước",
    "bh": "bây giờ",
}

def clean_teencode(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    
    # BƯỚC 1: Replace teencode TRƯỚC (case-insensitive)
    cleaned = text
    for slang, correct in TEENCODE_MAPPING.items():
        # Dùng regex để match whole word (tránh thay nhầm)
        import re
        pattern = r'\b' + re.escape(slang) + r'\b'
        cleaned = re.sub(pattern, correct, cleaned, flags=re.IGNORECASE)
    
    # BƯỚC 2: Model sửa lỗi chính tả (optional - có thể bỏ nếu model không tốt)
    # result = corrector(cleaned, max_new_tokens=256)[0]['generated_text']
    # cleaned = result.strip()
    
    return cleaned

# Test
# test_sent = "Anh ấy đã khiến nh th mất não đánh đồng giữa cos và wibu 🎉"
tens_sent = "gato quá hả cu bớt chen đi cho đời nó trong mặt m cx khác mẹ j đâu cùng 1 lò hết 😮‍💨😮‍💨😮‍💨"
print(f"Original: {tens_sent}")
print(f"Cleaned : {clean_teencode(tens_sent)}")

SyntaxError: unterminated string literal (detected at line 50) (518438738.py, line 50)

# Kappa

In [ ]:
import pandas as pd

# 1. Đọc dữ liệu từ 2 file
df1 = pd.read_csv("ket_qua_nop_bai.xlsx - Sheet1.csv")
df2 = pd.read_csv("kiet_final.xlsx - Worksheet.csv")

# 2. Lọc bỏ các dòng không có nhãn (nếu có)
df1 = df1.dropna(subset=['label_final'])
df2 = df2.dropna(subset=['label'])

# 3. Đổi tên cột nhãn để dễ phân biệt khi gộp
# File 1: label_final -> Label_File_1
# File 2: label -> Label_File_2
df1_sub = df1[['id', 'raw_comment', 'label_final']].rename(columns={'label_final': 'Label_File_1'})
df2_sub = df2[['id', 'label']].rename(columns={'label': 'Label_File_2'})

# 4. Gộp 2 file dựa trên ID chung
merged_df = pd.merge(df1_sub, df2_sub, on='id', how='inner')

# 5. Lọc ra các dòng có nhãn KHÁC nhau
disagreements = merged_df[merged_df['Label_File_1'] != merged_df['Label_File_2']]

# 6. Xuất kết quả
print(f"Tổng số comment khác nhãn: {len(disagreements)}")
print(disagreements[['raw_comment', 'Label_File_1', 'Label_File_2']].head(10))

# (Tùy chọn) Lưu ra file CSV
disagreements.to_csv('cac_comment_khac_nhau.csv', index=False)

🚀 Bắt đầu xử lý...
📖 Đang đọc file CSV: thanhthien_IAA_set_500_samples.csv
📖 Đang đọc file CSV: Huy_IAA_set_500_samples.csv
📖 Đang đọc file Excel: IAA_set_500_samples.xlsx
🔄 Đang gộp dữ liệu...
📊 Tổng số mẫu ghép được: 499
------------------------------
✅ Số lượng mẫu hợp lệ: 294
------------------------------
🔹 KẾT QUẢ PAIRWISE KAPPA:
   - Rater 1 vs Rater 2: 0.3393
   - Rater 1 vs Rater 3: 0.5229
   - Rater 2 vs Rater 3: 0.3136
   => Trung bình: 0.3920
------------------------------
🏆 FLEISS' KAPPA: 0.3822
------------------------------
📝 Phát hiện 156 trường hợp lệch nhãn.
💾 Đã lưu file tại: ket_qua_lech_nhan.csv


# Convert json to xlsv


In [3]:
import pandas as pd

# Thay thế pd.read_csv bằng pd.read_excel
# Lưu ý: Bạn cần cài thư viện openpyxl nếu chưa có: pip install openpyxl
df1 = pd.read_excel("ket_qua_nop_bai.xlsx") 
df2 = pd.read_excel("kiet_final.xlsx")

# Các bước xử lý tiếp theo giữ nguyên
# 1. Lọc bỏ dòng thiếu nhãn
df1 = df1.dropna(subset=['label_final'])
df2 = df2.dropna(subset=['label'])

# 2. Đổi tên cột để gộp
df1_sub = df1[['id', 'raw_comment', 'label_final']].rename(columns={'label_final': 'Label_File_1'})
df2_sub = df2[['id', 'label']].rename(columns={'label': 'Label_File_2'})

# 3. Gộp dữ liệu
merged_df = pd.merge(df1_sub, df2_sub, on='id', how='inner')

# 4. Tìm các comment khác nhau
disagreements = merged_df[merged_df['Label_File_1'] != merged_df['Label_File_2']]

# 5. Xuất kết quả
print(f"Tổng số comment khác nhãn: {len(disagreements)}")
# Lưu ra file Excel mới thay vì CSV cho tiện theo dõi
disagreements.to_excel('cac_comment_khac_nhau.xlsx', index=False)

Tổng số comment khác nhãn: 141


In [3]:
!pip install openpyxl



   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpy